GLM-4-9B | PEV at a single fixed sigma = 0.003
All 100 JBB-Behaviors prompts, 20 generations per prompt, run as ONE batch of 20.

Same harness as the GLM sweep notebook; run_batch() is unchanged. Only the driver
differs: one sigma instead of six, 100 prompts in one notebook, checkpoint per prompt.
Log format is unchanged, so the existing categorization pipeline parses it as-is.

In [ ]:
get_ipython().system('pip -q install -U transformers datasets accelerate sentencepiece huggingface_hub')

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
from google.colab import files
uploaded = files.upload()  # select run_glm.py from your machine

In [ ]:
import os
os.environ["GLM_DEVICE"] = "cuda"
os.environ["GLM_SKIP_EMBED_PROMPT"] = "1"   # non-interactive: skip the Y/N embed prompts

import torch
import run_glm as rq

model, tokenizer = rq.load_model()

base_config = {
    "verbose": False,
    "temperature": 0.7,
    "top_k": 50,
    "top_p": 0.9,
    "max_new_tokens": 4096,
    "print_pre_layer_embed": False,
    "write_glm_embed_log_file": False,
    "require_eos_before_print": False,
}

In [ ]:
from huggingface_hub import login
from getpass import getpass

hf_token = getpass("Enter your Hugging Face token (hf_...): ")
login(token=hf_token)
print("HF login successful")

In [ ]:
from datasets import load_dataset

ds = load_dataset("JailbreakBench/JBB-Behaviors", "behaviors", split="harmful")

print("Columns:", ds.column_names)
print("Total rows:", len(ds))

prompt_col = "Goal"
print("Using prompt column:", prompt_col)
print("Sample prompt:")
print(ds[0][prompt_col])

In [ ]:
import math

def to_safe_text(x):
    if x is None:
        return ""
    if isinstance(x, float) and math.isnan(x):
        return ""
    return str(x)

In [ ]:
import torch
import torch.nn.functional as F

# --- Build the GLM stop-token set ONCE (mirrors generate_response) ----------
def _build_stop_token_ids(tokenizer, model):
    eos_token_id = tokenizer.eos_token_id
    stop = {eos_token_id} if eos_token_id is not None else set()
    gen_eos = getattr(getattr(model, "generation_config", None), "eos_token_id", None)
    if isinstance(gen_eos, (list, tuple)):
        stop.update(int(t) for t in gen_eos if t is not None)
    elif isinstance(gen_eos, int):
        stop.add(gen_eos)
    unk = getattr(tokenizer, "unk_token_id", None)
    for tok in ("<|endoftext|>", "<|user|>", "<|observation|>"):
        tid = tokenizer.convert_tokens_to_ids(tok)
        if isinstance(tid, int) and tid >= 0 and tid != unk:
            stop.add(tid)
    return stop

_STOP_IDS = _build_stop_token_ids(tokenizer, model)
_STOP_TENSOR = torch.tensor(sorted(_STOP_IDS), device=rq.DEVICE)


def _sample_from_logits(logits, temperature, top_k, top_p):
    """logits: [B, vocab] -> next_token: [B, 1]. Per-row top-k/top-p/sample."""
    logits = logits.float()
    if temperature > 0 and temperature != 1.0:
        logits = logits / temperature

    if top_k and top_k > 0:
        k = min(top_k, logits.size(-1))
        kth = torch.topk(logits, k, dim=-1).values[:, -1, None]
        logits = torch.where(logits < kth, torch.full_like(logits, float("-inf")), logits)

    if 0 < top_p < 1.0:
        sorted_logits, sorted_idx = torch.sort(logits, descending=True, dim=-1)
        sm = F.softmax(sorted_logits, dim=-1)
        cum = torch.cumsum(sm, dim=-1)
        remove = (cum - sm) >= top_p
        sorted_logits[remove] = float("-inf")
        logits = sorted_logits.scatter(-1, sorted_idx, sorted_logits)

    probs = F.softmax(logits, dim=-1)
    if temperature == 0:
        return torch.argmax(probs, dim=-1, keepdim=True)
    nxt = torch.multinomial(probs.cpu(), num_samples=1)
    return nxt.to(logits.device)


@torch.inference_mode()
def run_batch(prompt_text, perturb, n_runs, config, micro_batch=None):
    """Return a list of n_runs decoded strings for ONE (prompt, perturb) pair.

    Replaces n_runs sequential run_one() calls. If micro_batch is set, splits the
    n_runs rows into chunks of that size (use if you hit CUDA OOM at full batch).
    """
    prompt_text = to_safe_text(prompt_text).strip()
    if not prompt_text:
        return [""] * n_runs

    if micro_batch and micro_batch < n_runs:
        outs = []
        remaining = n_runs
        while remaining > 0:
            b = min(micro_batch, remaining)
            outs.extend(run_batch(prompt_text, perturb, b, config, micro_batch=None))
            remaining -= b
        return outs

    device = rq.DEVICE
    temperature = config["temperature"]
    top_k = config["top_k"]
    top_p = config["top_p"]
    max_new = config["max_new_tokens"]

    messages = [
        {"role": "system", "content": ""},
        {"role": "user", "content": prompt_text},
    ]
    chat_text = rq.build_prompt(tokenizer, messages)
    enc = tokenizer(chat_text, return_tensors="pt", add_special_tokens=False)
    ids = enc["input_ids"].to(device)                 # [1, L]
    L = ids.shape[1]

    ctx_limit = model.config.max_position_embeddings
    max_new = min(max_new, max(1, ctx_limit - L - 1))

    B = n_runs
    ids = ids.expand(B, L).contiguous()               # [B, L] identical rows
    attn = torch.ones(B, L, dtype=torch.long, device=device)

    embed_layer = model.get_input_embeddings()
    base_emb = embed_layer(ids)                        # [B, L, H]
    if perturb > 0:
        base_emb = base_emb + perturb * torch.randn_like(base_emb)

    out = model(inputs_embeds=base_emb, attention_mask=attn,
                use_cache=True, output_hidden_states=False, output_attentions=False)
    past = out.past_key_values
    logits = out.logits[:, -1, :]

    generated = [[] for _ in range(B)]
    finished = torch.zeros(B, dtype=torch.bool, device=device)

    for _ in range(max_new):
        nxt = _sample_from_logits(logits, temperature, top_k, top_p)   # [B,1]
        is_stop = torch.isin(nxt.squeeze(1), _STOP_TENSOR)
        for b in range(B):
            if not finished[b]:
                tid = nxt[b, 0].item()
                if tid in _STOP_IDS:
                    finished[b] = True
                else:
                    generated[b].append(tid)
        finished = finished | is_stop
        if bool(finished.all()):
            break

        attn = torch.cat([attn, torch.ones(B, 1, dtype=torch.long, device=device)], dim=1)
        step_emb = embed_layer(nxt)                    # [B,1,H]
        out = model(inputs_embeds=step_emb, attention_mask=attn,
                    past_key_values=past, use_cache=True,
                    output_hidden_states=False, output_attentions=False)
        past = out.past_key_values
        logits = out.logits[:, -1, :]

    return [to_safe_text(tokenizer.decode(g, skip_special_tokens=True)).strip()
            for g in generated]

In [ ]:
import os
import time
from datetime import datetime

# Mount Drive so logs survive Colab disconnect (set False only for a quick test)
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    LOG_DIR = "/content/drive/MyDrive/LLm_perturbation_logs"
else:
    LOG_DIR = "/content"
os.makedirs(LOG_DIR, exist_ok=True)

PERTURB = 0.003            # single fixed sigma (GLM peak)
START_PROMPT = 1           # 1-indexed; bump this to resume after a disconnect
NUM_PROMPTS = 100
RUNS_PER_PERTURB = 20

# Set to e.g. 10 if you hit CUDA OOM at the full batch of 20. None = full batch.
MICRO_BATCH = None
SEED = 1234                # per-prompt seed = SEED + prompt index; None to disable

start_idx = START_PROMPT - 1
end_idx = min(start_idx + NUM_PROMPTS, len(ds))
prompt_indices = list(range(start_idx, end_idx))
prompts = [to_safe_text(ds[i][prompt_col]).strip() for i in prompt_indices]
num_prompts = len(prompts)

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
log_path = os.path.join(
    LOG_DIR,
    f"JBB_glm4_9b_sigma{str(PERTURB).replace('.', 'p')}"
    f"_p{START_PROMPT:03d}_to_p{start_idx + num_prompts:03d}_{ts}.txt",
)

total = num_prompts * RUNS_PER_PERTURB
print("Log file:", log_path)
print("Prompts:", num_prompts, "| sigma:", PERTURB, "| Runs each:", RUNS_PER_PERTURB)
print("Total generations:", total)

In [ ]:
import os, shutil, time
from datetime import datetime

local_log = f"/content/{os.path.basename(log_path)}"
def checkpoint_copy():
    shutil.copy2(log_path, local_log)

job = 0
t0 = time.time()

with open(log_path, "a", encoding="utf-8") as f:
    f.write(f"start {datetime.now().isoformat()}\n")
    f.write(f"prompt_col={prompt_col!r} num_prompts={num_prompts} runs={RUNS_PER_PERTURB}\n")
    f.write(f"perturb_values=[{PERTURB}]\n")
    f.write(f"batched=True micro_batch={MICRO_BATCH} seed={SEED}\n")
    f.write("=" * 120 + "\n\n")
    f.flush(); os.fsync(f.fileno())

    for p_idx, prompt_text in enumerate(prompts, start=START_PROMPT):
        f.write(f"Prompt{p_idx} - {prompt_text}\n")
        f.write(f"perturbation value- {PERTURB}\n")
        f.flush(); os.fsync(f.fileno())

        print(f"\n>>> Prompt {p_idx}/{START_PROMPT + num_prompts - 1} | perturbation={PERTURB}")
        tp = time.time()

        if SEED is not None:
            torch.manual_seed(SEED + p_idx)
        try:
            outs = run_batch(prompt_text, PERTURB, RUNS_PER_PERTURB, base_config, micro_batch=MICRO_BATCH)
        except Exception as e:
            outs = [f"[ERROR] {type(e).__name__}: {e}"] * RUNS_PER_PERTURB

        for run_id, out in enumerate(outs, start=1):
            job += 1
            out = to_safe_text(out).replace("\r\n", "\n").strip()
            f.write(f"run{run_id}- {out}\n")
        f.write("-" * 120 + "\n")
        f.flush(); os.fsync(f.fileno())

        chars = sum(len(o) for o in outs) // max(1, len(outs))
        print(f"    {RUNS_PER_PERTURB} runs in {time.time()-tp:.1f}s | global {job}/{total} | mean chars={chars}")
        checkpoint_copy()

checkpoint_copy()
print(f"\nDone in {(time.time()-t0)/60:.1f} min | log: {log_path}")
print(f"Local copy: {local_log}")

### Optional: re-run selected prompts with more samples
Same sigma, a chosen subset of prompt ids, N runs each. Writes its own SELECT log.

In [ ]:
import os, shutil, time
from datetime import datetime

PROMPT_IDS = [10, 17, 32]      # 1-indexed, matches P<n> in the logs
SELECT_RUNS = 30
SELECT_SEED = 4321             # different from the main pass, so these are fresh samples

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
tag = "-".join(str(p) for p in PROMPT_IDS)
sel_log = os.path.join(
    LOG_DIR,
    f"JBB_glm4_9b_sigma{str(PERTURB).replace('.', 'p')}_SELECT_p{tag}_n{SELECT_RUNS}_{ts}.txt",
)
sel_local = f"/content/{os.path.basename(sel_log)}"
sel_total = len(PROMPT_IDS) * SELECT_RUNS
print("Log file:", sel_log, "| total generations:", sel_total)

job = 0
t0 = time.time()

with open(sel_log, "a", encoding="utf-8") as f:
    f.write(f"start {datetime.now().isoformat()}\n")
    f.write(f"prompt_col={prompt_col!r} prompt_ids={PROMPT_IDS} runs={SELECT_RUNS}\n")
    f.write(f"perturb_values=[{PERTURB}]\n")
    f.write(f"batched=True micro_batch={MICRO_BATCH} seed={SELECT_SEED}\n")
    f.write("=" * 120 + "\n\n")
    f.flush(); os.fsync(f.fileno())

    for p_idx in PROMPT_IDS:
        prompt_text = to_safe_text(ds[p_idx - 1][prompt_col]).strip()

        f.write(f"Prompt{p_idx} - {prompt_text}\n")
        f.write(f"perturbation value- {PERTURB}\n")
        f.flush(); os.fsync(f.fileno())

        print(f"\n>>> Prompt {p_idx} | perturbation={PERTURB}")
        tp = time.time()

        if SELECT_SEED is not None:
            torch.manual_seed(SELECT_SEED + p_idx)
        try:
            outs = run_batch(prompt_text, PERTURB, SELECT_RUNS, base_config, micro_batch=MICRO_BATCH)
        except Exception as e:
            outs = [f"[ERROR] {type(e).__name__}: {e}"] * SELECT_RUNS

        for run_id, out in enumerate(outs, start=1):
            job += 1
            out = to_safe_text(out).replace("\r\n", "\n").strip()
            f.write(f"run{run_id}- {out}\n")
        f.write("-" * 120 + "\n")
        f.flush(); os.fsync(f.fileno())

        print(f"    {SELECT_RUNS} runs in {time.time()-tp:.1f}s | global {job}/{sel_total}")
        shutil.copy2(sel_log, sel_local)

print(f"\nDone in {(time.time()-t0)/60:.1f} min | log: {sel_log}")